In [1]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from statsmodels.tsa.stattools import pacf
import numpy as np

In [2]:
import os
import sys
current_dir = os.path.dirname(os.path.realpath(__name__))
parent_dir = os.path.dirname(current_dir)
print(parent_dir)
sys.path.append(parent_dir)
from src.data.data_loader import DataLoader
dl = DataLoader()
dl.prepare_df()
prepared_df = dl.prepared_df
dl.train_test_split()

C:\Users\jonah.eisen\dsi\global-stock-index-ml-classification


In [3]:
dl.X_train.head()

,Last Adj Close,Volume,LowProportion,HighProportion
Date,,,,
2003-01-10,5210.339844,1.477200e+09,0.991511,1.004248
2003-01-13,5209.799805,1.371500e+09,0.995975,1.006883
2003-01-14,5209.209961,1.355600e+09,0.996499,1.004696
2003-01-15,5233.660156,1.385200e+09,0.986618,1.000290
2003-01-16,5171.450195,1.534600e+09,0.996568,1.006555


In [4]:
train_pacf = pacf(dl.y_train.rolling(window=10).mean().dropna(), method='ywm')
alpha = 0.1
ar_terms = [i for i,val in enumerate(np.abs(train_pacf) > alpha) if val]
ar_terms = ar_terms[1:]
ar_terms

[1, 2, 3, 4, 6, 8, 9, 10, 11, 12, 20, 21, 22, 30, 31]

In [5]:
prepared_df.head()

,Last Close,Close,Last Adj Close,Volume,LowProportion,HighProportion,Delta Close
Date,,,,,,,
2003-01-10,5210.040039,5209.209961,5210.339844,1.477200e+09,0.991511,1.004248,-0.830078
2003-01-13,5209.799805,5233.660156,5209.799805,1.371500e+09,0.995975,1.006883,23.860351
2003-01-14,5209.209961,5171.450195,5209.209961,1.355600e+09,0.996499,1.004696,-37.759766
2003-01-15,5233.729980,5165.339844,5233.660156,1.385200e+09,0.986618,1.000290,-68.390136
2003-01-16,5171.450195,5108.509766,5171.450195,1.534600e+09,0.996568,1.006555,-62.940429


In [6]:
p = max(ar_terms)
lag_dict = {}
for j in ar_terms:
    lag_dict[f'y_(t-{j})'] = []
for i in range(p,prepared_df.shape[0]):
    for j in ar_terms:
        lag_val = prepared_df['Delta Close'].iloc[i-j]
        lag_dict[f'y_(t-{j})'].append(lag_val) 

In [7]:
cropped_df = prepared_df.iloc[p:,:]
cropped_df.shape

(4596, 7)

In [8]:
cropped_df = cropped_df.assign(**lag_dict)
cropped_df.shape

(4596, 22)

In [9]:
delta_close = cropped_df['Delta Close']
y = DataLoader.quantize_delta_close(delta_close)
cropped_df.drop(columns=['Last Close', 'Close', 'Delta Close'],inplace=True)

In [12]:
len(lag_dict.keys())

15

In [13]:
val_size = 0.2
X_train, X_test = DataLoader.time_split_2D(cropped_df)
X_train, X_val = DataLoader.time_split_2D(X_train)
y_train, y_test = DataLoader.time_split_1D(y)
y_train, y_val = DataLoader.time_split_1D(y_train)

In [19]:
rf = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)
y_prob = rf.predict_proba(X_val)[:,1]



In [20]:
print("Random Forest Accuracy:", rf.score(X_val, y_val))
print(classification_report(y_val, y_pred))
print("ROC-AUC:", roc_auc_score(y_val, y_prob))

Random Forest Accuracy: 0.6779891304347826
              precision    recall  f1-score   support

           0       0.69      0.56      0.61       340
           1       0.67      0.78      0.72       396

    accuracy                           0.68       736
   macro avg       0.68      0.67      0.67       736
weighted avg       0.68      0.68      0.67       736

ROC-AUC: 0.7509655377302437
